### BLS/OEWS data provenance

The occupational employment and wage data used in this analysis were obtained
from the U.S. Bureau of Labor Statistics Occupational Employment and Wage
Statistics (OEWS) national data tables.

The analysis uses the May national OEWS releases for:

- Pre-period: 2017, 2018, and 2019
- Post-period: 2023, 2024, and 2025

The original national XLSX files were downloaded from the official BLS OEWS
tables archive and preserved locally under:

`08_Labor_Outcomes/BLS/Input/`

All subsequent processing reads from these preserved local copies rather than
retrieving files dynamically from the BLS website.

**Official source:** [BLS Occupational Employment and Wage Statistics (OEWS) Tables](https://www.bls.gov/oes/tables.htm)



# Occupational Employment and Wage Outcome Construction

## Analytical objective

Construct comparable SOC6-level measures of occupational employment and wage trajectories for the RQ1 outcome models.

## Measurement design

The analysis uses national Occupational Employment and Wage Statistics (OEWS) estimates for two periods:

* **Pre-period:** May 2017–2019
* **Post-period:** May 2023–2025

Each occupation’s annual estimates are averaged within the pre- and post-periods. The primary trajectory measure is the annualized log change between the two period averages:

$$
\Delta Y_o
=
\frac{100}{6}
\left[
\ln(\overline{Y}_{o,post})
-
\ln(\overline{Y}_{o,pre})
\right]
$$

where \(Y\) represents occupational employment or real median wage, and six is the interval between the period midpoints, 2018 and 2024.

Employment is measured using national OEWS wage-and-salary employment (`TOT_EMP`). Wages are converted to May 2025 dollars using the CPI-U before calculating wage change. The final choice between hourly and annual median wage is determined from the cross-year coverage audit; the two measures are not mixed.

## Comparability rule

The frozen configuration–exposure universe contains 604 SOC6 occupations. Labor outcomes are constructed only where the historical and current SOC definitions are sufficiently comparable. Employment and wage models may therefore contain different numbers of occupations because employment counts can sometimes be aggregated across predecessor codes, whereas median wages cannot be reconstructed from component medians.

These measures describe occupational trajectories. They do not, by themselves, identify causal effects of GenAI.


Setup and occupation universe

In [ ]:
# Setup and occupation universe

from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path(
    "/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes"
)

BLS_DIR = BASE_DIR / "08_Labor_Outcomes/Input/BLS"
MB3_DIR = BASE_DIR / "08_Labor_Outcomes/Input/BLS_MB3"
CROSSWALK_DIR = BASE_DIR / "08_Labor_Outcomes/Input/SOC_Crosswalk"
outcome_dir = BASE_DIR / "08_Labor_Outcomes/Output"
RQ1_FILE = (
    BASE_DIR
    / "07_GenAI_Exposure/Output/Occupational_Configuration_Exposure_v1.csv"
)

pre_years = [2017, 2018, 2019]
post_years = [2023, 2024, 2025]
analysis_years = pre_years + post_years
YEAR_GAP = np.mean(post_years) - np.mean(pre_years)

bls_files = {
    year: (
        MB3_DIR / f"oes_mb3_{year}a.xlsx"
        if year in pre_years
        else BLS_DIR / f"national_M{year}_dl.xlsx"
    )
    for year in analysis_years
}

rq1_sample = pd.read_csv(RQ1_FILE, dtype={"soc6": str})
target_socs = sorted(rq1_sample["soc6"].dropna().unique())

print(f"Target occupations: {len(target_socs)}")

Load the annual BLS data

In [ ]:
# Standardize column names across the six BLS workbooks.
def normalize_column(name):
    return "".join(c for c in str(name).upper() if c.isalnum())

bls_columns = {
    "AREA": "area",
    "NAICS": "naics",
    "IGROUP": "industry_group",
    "OGROUP": "occupation_group",
    "OCCGROUP": "occupation_group",
    "OCCCODE": "source_soc",
    "SOCCODE": "source_soc",
    "TOTEMP": "employment",
    "AMEAN": "annual_mean",
    "AMEDIAN": "annual_median",
}

outcome_source_frames = {}

for year, path in bls_files.items():
    raw = pd.read_excel(
        path,
        sheet_name=0,
        dtype=str,
        usecols=lambda column: normalize_column(column) in bls_columns,
    )
    raw.columns = [bls_columns[normalize_column(c)] for c in raw.columns]

    # Retain national, cross-industry, detailed occupations.
    for column, value in {"area": 99, "naics": 0}.items():
        if column in raw:
            raw = raw.loc[
                pd.to_numeric(raw[column], errors="coerce").eq(value)
            ]

    if "industry_group" in raw:
        raw = raw.loc[
            raw["industry_group"].str.strip().str.lower().eq("cross-industry")
        ]

    raw = raw.loc[
        raw["occupation_group"].str.strip().str.lower().eq("detailed"),
        ["source_soc", "employment", "annual_mean", "annual_median"],
    ].copy()

    raw["source_soc"] = raw["source_soc"].str.extract(
        r"(\d{2}-\d{4})", expand=False
    )
    raw = raw.dropna(subset=["source_soc"])

    for column in ["employment", "annual_mean", "annual_median"]:
        raw[column] = pd.to_numeric(
            raw[column].str.replace(",", "", regex=False).str.strip(),
            errors="coerce",
        )

    outcome_source_frames[year] = raw.set_index(
        "source_soc", verify_integrity=True
    )
    print(f"{year}: {len(raw)} occupations loaded")

SOC crosswalks

In [ ]:
soc_mappings = {}

for period, filename, sheet, header, source_column in [
    (
        "soc2010", "soc_2010_to_2018_crosswalk.xlsx",
        "Sorted by 2010", 8, "2010 SOC Code"
    ),
    (
        "hybrid2019", "oes_2019_hybrid_structure.xlsx",
        "OES2019 Hybrid", 5, "OES 2019 Estimates Code"
    ),
]:
    pairs = pd.read_excel(
        CROSSWALK_DIR / filename,
        sheet_name=sheet,
        header=header,
        dtype=str,
        usecols=lambda c: str(c).strip() in [
            source_column, "2018 SOC Code"
        ],
    )
    pairs.columns = pairs.columns.str.strip()
    pairs = pairs[[source_column, "2018 SOC Code"]].copy()
    pairs.columns = ["source_soc", "soc6"]

    pairs = (
        pairs.apply(
            lambda column: column.str.extract(r"(\d{2}-\d{4})", expand=False)
        )
        .dropna()
        .drop_duplicates()
    )

    soc_mappings[period] = {
        "sources": pairs.groupby("soc6")["source_soc"].agg(sorted).to_dict(),
        "target_counts": pairs.groupby("source_soc")["soc6"].nunique().to_dict(),
    }

Construct the harmonized annual panel

In [ ]:
annual_rows = []

for soc6 in target_socs:
    for year in analysis_years:
        frame = outcome_source_frames[year]

        if year in post_years:
            sources = [soc6]
            mapping_method = "direct_2018_soc"
            emp_mean_ok = median_ok = True
        else:
            mapping = soc_mappings[
                "hybrid2019" if year == 2019 else "soc2010"
            ]
            sources = mapping["sources"].get(soc6, [])
            exclusive = all(
                mapping["target_counts"][code] == 1 for code in sources
            )

            if not sources:
                mapping_method = "no_official_mapping"
            elif exclusive:
                mapping_method = (
                    "one_to_one" if len(sources) == 1
                    else "multiple_sources_to_one_target"
                )
            else:
                mapping_method = (
                    "one_source_to_multiple_targets" if len(sources) == 1
                    else "complex_many_to_many"
                )

            emp_mean_ok = (
                bool(sources)
                and exclusive
                and all(code in frame.index for code in sources)
            )
            median_ok = emp_mean_ok and len(sources) == 1

        values = frame.reindex(sources)
        employment = annual_mean = annual_median = np.nan

        if emp_mean_ok and sources and values["employment"].gt(0).all():
            employment = values["employment"].sum()

        # A single-source mean does not require employment.
        if emp_mean_ok and len(sources) == 1:
            annual_mean = values["annual_mean"].iloc[0]
        elif (
            emp_mean_ok
            and pd.notna(employment)
            and values["annual_mean"].notna().all()
        ):
            annual_mean = np.average(
                values["annual_mean"], weights=values["employment"]
            )

        if median_ok and len(sources) == 1:
            annual_median = values["annual_median"].iloc[0]

        annual_rows.append({
            "soc6": soc6,
            "year": year,
            "mapping_method": mapping_method,
            "source_codes": " | ".join(sources),
            "source_count": len(sources),
            "employment": employment,
            "annual_mean_nominal": annual_mean,
            "annual_median_nominal": annual_median,
            "emp_mean_structural_ok": bool(emp_mean_ok),
            "median_structural_ok": bool(median_ok),
        })

annual_outcome_panel = pd.DataFrame(annual_rows)

In [ ]:
cpi_may = {
    2017: 244.733,
    2018: 251.588,
    2019: 256.092,
    2023: 304.127,
    2024: 314.069,
    2025: 321.465,
}

annual_outcome_panel["cpi_may"] = annual_outcome_panel["year"].map(cpi_may)

for wage in ["mean", "median"]:
    annual_outcome_panel[f"annual_{wage}_real_2025"] = (
        annual_outcome_panel[f"annual_{wage}_nominal"]
        * cpi_may[2025]
        / annual_outcome_panel["cpi_may"]
    )

Pre/post averages and annualized growth

**mean_wage_real means** inflation-adjusted mean wage. It forms the column names mean_wage_real_pre_avg and mean_wage_real_post_avg.

**annual_mean_sample** indicates whether the occupation has valid mean-wage data for all six years: True means eligible for the wage-growth calculation; False means excluded.

In [ ]:
outcome_specs = {
    "employment": ("employment", "employment", "employment_sample"),
    "mean_wage": ("annual_mean_real_2025", "mean_wage_real", "annual_mean_sample"),
    "median_wage": ("annual_median_real_2025", "median_wage_real", "annual_median_sample"),
}

occupational_outcomes = pd.DataFrame(index=pd.Index(target_socs, name="soc6"))

for outcome, (column, prefix, sample_flag) in outcome_specs.items():
    annual = annual_outcome_panel.pivot(
        index="soc6", columns="year", values=column
    ).reindex(index=target_socs, columns=analysis_years)

    pre = annual[pre_years].mean(axis=1)
    post = annual[post_years].mean(axis=1)
    complete = annual.notna().all(axis=1)

    if outcome == "mean_wage":
        complete &= np.isfinite(annual).all(axis=1) & annual.gt(0).all(axis=1)

    valid = complete & pre.gt(0) & post.gt(0)
    change = (np.log(post.where(valid)) - np.log(pre.where(valid))) / YEAR_GAP

    occupational_outcomes[f"{prefix}_pre_avg"] = pre
    occupational_outcomes[f"{prefix}_post_avg"] = post
    occupational_outcomes[sample_flag] = complete
    occupational_outcomes[f"{outcome}_ann_log_change"] = change
    occupational_outcomes[f"{outcome}_ann_log_pct"] = 100 * change

output_columns = (
    [f"{prefix}_{period}_avg"
     for period in ["pre", "post"]
     for _, prefix, _ in outcome_specs.values()]
    + [flag for _, _, flag in outcome_specs.values()]
    + [f"{outcome}_ann_log_{suffix}"
       for suffix in ["change", "pct"]
       for outcome in outcome_specs]
)
occupational_outcomes = occupational_outcomes[output_columns].reset_index()

In [ ]:
annual_outcome_panel.to_csv(
    outcome_dir / "OEWS_Annual_Harmonized_Panel_v1.csv",
    index=False,
)
occupational_outcomes.to_csv(
    outcome_dir / "OEWS_Occupational_Outcomes_v1.csv",
    index=False,
)